# Hand-Bill Detection v2 — Colab Training

Trains a YOLO11 model on the **`bill_hand.v2i.yolov11.zip`** dataset
(`bill-hand` = hand holding a bill, `no_bill` = hand with no bill).

**Setup:** Runtime → Change runtime type → **T4 GPU**.

**Steps:** run cells in order → upload the zip when asked → download `best.pt` at the end.

In [ ]:
# 1) Install Ultralytics
!pip install -q ultralytics
import ultralytics
print('Ultralytics', ultralytics.__version__)

## 2) Upload the dataset

Upload `bill_hand.v2i.yolov11.zip` (from your local Downloads folder) when the picker opens.

In [ ]:
from google.colab import files
import zipfile, glob, os

uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

!rm -rf /content/dataset
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')
print('Extracted to /content/dataset')

tr = glob.glob('/content/dataset/train/images/*')
va = glob.glob('/content/dataset/valid/images/*')
te = glob.glob('/content/dataset/test/images/*')
print(f'train: {len(tr)} | valid: {len(va)} | test: {len(te)}')
print('data.yaml:')
print(open('/content/dataset/data.yaml').read())

## 3) Training configuration

Adjust if you like. `yolo11n` is the fastest; `yolo11s` is a bit more accurate.

In [ ]:
EPOCHS = 100
IMGSZ = 640
BATCH = 32
MODEL = 'yolo11n.pt'
DATA = '/content/dataset/data.yaml'
RUN_NAME = 'bill_hand_v2'

In [ ]:
# 4) Train
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project='/content/runs',
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    patience=30,
)

## 5) Results summary

Run this after the training cell above finishes.

In [ ]:
import pandas as pd, os, glob
run_dir = f'/content/runs/{RUN_NAME}'
found = sorted(glob.glob(f'/content/runs/{RUN_NAME}*'))
if not found:
    raise RuntimeError('No training run found. Did the training cell run and finish?\n'
                       f'Runs present: {glob.glob("/content/runs/*")}')
if not os.path.exists(run_dir):
    run_dir = found[0]
    print('Using run dir:', run_dir)
df = pd.read_csv(f'{run_dir}/results.csv')
print('Last 5 rows:')
print(df.tail())
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"\nBest epoch: {int(best['epoch'])}  mAP50={best['metrics/mAP50(B)']:.4f}  mAP50-95={best['metrics/mAP50-95(B)']:.4f}")
print(f"Final:        mAP50={df['metrics/mAP50(B)'].iloc[-1]:.4f}  mAP50-95={df['metrics/mAP50-95(B)'].iloc[-1]:.4f}")
print('\nRun dir:', sorted(os.listdir(run_dir)))

In [ ]:
# 6) Validate on the held-out TEST split (independent check)
from ultralytics import YOLO
best = YOLO(f'/content/runs/{RUN_NAME}/weights/best.pt')
r = best.val(data=DATA, split='test', imgsz=IMGSZ, verbose=False)
print(f"TEST set  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}  precision={r.box.mp:.4f}  recall={r.box.mr:.4f}")
per_class = r.box.maps
print('per-class mAP50:', dict(zip(best.names.values(), [round(m,4) for m in per_class])))

In [ ]:
# 7) Download best.pt + training curves
from google.colab import files
run_dir = f'/content/runs/{RUN_NAME}'
files.download(f'{run_dir}/weights/best.pt')
files.download(f'{run_dir}/results.png')

## 8) Quick sanity prediction on a test image
```python
best = YOLO('/content/runs/detect/bill_hand_v2/weights/best.pt')
best.predict('/content/dataset/test/images', conf=0.25, save=True, project='/content/test_pred', name='out')
```